In [ ]:
# SPDX-FileCopyrightText: 2024 Dan J. Bower <dbower@eaps.ethz.ch>
#
# SPDX-License-Identifier: GPL-3.0-or-later

import importlib.resources
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    PurePhase,
    ThermodynamicState,
    debug_logger,
)
from atmodeller.eos import DATA_DIRECTORY, ZhangDuanMixture

ZHANG_DUAN_DIRECTORY = Path("zhang_duan_2009")

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

# Zhang and Duan (2009)

This notebook is available at `notebooks/examples/zhang_duan.ipynb` and is easiest to obtain by downloading the source code. This notebook reproduces the results of Zhang and Duan (2009), Geochimica et Cosmochimica Acta.

First, initialize the mixture model with the species to include.

In [ ]:
# Define the species we want to include in our model, using Hill notation for the chemical formulas
species: tuple[str, ...] = ("H2O", "CO2", "CH4", "O2", "CO", "H2", "C2H6")

# Construct the mixture EOS for each species using the Zhang and Duan (2009) model
eos_H2O = ZhangDuanMixture(species, "H2O")
eos_CO2 = ZhangDuanMixture(species, "CO2")
eos_CH4 = ZhangDuanMixture(species, "CH4")
eos_O2 = ZhangDuanMixture(species, "O2")
eos_CO = ZhangDuanMixture(species, "CO")
eos_H2 = ZhangDuanMixture(species, "H2")
eos_C2H6 = ZhangDuanMixture(species, "C2H6")

We can now specify these EOS as activity models.

In [ ]:
H2O_g = ChemicalSpecies.create_gas("H2O", activity=eos_H2O)
CO2_g = ChemicalSpecies.create_gas("CO2", activity=eos_CO2)
CH4_g = ChemicalSpecies.create_gas("CH4", activity=eos_CH4)
O2_g = ChemicalSpecies.create_gas("O2", activity=eos_O2)
CO_g = ChemicalSpecies.create_gas("CO", activity=eos_CO)
H2_g = ChemicalSpecies.create_gas("H2", activity=eos_H2)
C2H6_g = ChemicalSpecies.create_gas("C2H6", activity=eos_C2H6)

# NOTE: The ordering of the gas species here must correspond to the ordering of the species in the
# tuple used to initialize the ZhangDuanMixture EOS models above.
gas_species = (H2O_g, CO2_g, CH4_g, O2_g, CO_g, H2_g, C2H6_g)

The experiments in Zhang and Duan (2009) are buffered by graphite.

In [ ]:
graphite: PurePhase = PurePhase.from_species("C", state="s", solve_for_stability=False)
condensates: tuple[PurePhase, ...] = (graphite,)

Set the temperature and pressure conditions corresponding to Figure 2 and 3.

In [ ]:
pressure = 2.4e3 * 10  # bar
temperature = 1273  # K

Define and solve the system.

In [ ]:
mixture_eos_state = ThermodynamicState.from_species(
    gas_species, pressure=pressure, temperature=temperature, condensates=condensates
)

# Number of calculations
num = 100

# Specify bounds in X_O space (X_O = O / (H + O), with H = 1)
# O is then recovered via O = X_O / (1 - X_O)
X_O_min = 0.01
X_O_max = 0.99
X_O_space = np.linspace(X_O_min, X_O_max, num)
O_moles = X_O_space / (1 - X_O_space)

# Normalize moles to 1 mole of H
mole_constraints = {"H": 1.0, "O": O_moles}

model = EquilibriumModel.from_state(
    mixture_eos_state, mass_constraints=mole_constraints, mass_units="moles"
)

output = model.solve_with_default()

output.solver_stats_to_logger(logger)

Get output quantities for plotting.

In [ ]:
output_dict = output.to_dict()

moles_hydrogen = output_dict["gas"]["elements"]["number_moles"]["H"]
moles_oxygen = output_dict["gas"]["elements"]["number_moles"]["O"]

X_O = moles_oxygen / (moles_hydrogen + moles_oxygen)
X_H2O = output_dict["gas"]["species"]["mole_fraction"]["H2O_g"]
X_CH4 = output_dict["gas"]["species"]["mole_fraction"]["CH4_g"]
X_H2 = output_dict["gas"]["species"]["mole_fraction"]["H2_g"]
X_CO2 = output_dict["gas"]["species"]["mole_fraction"]["CO2_g"]

fO2 = output_dict["gas"]["species"]["activity"]["O2_g"]
fO2_dIW_1_bar = output_dict["gas"]["phase"]["log10dIW_1_bar"]
fO2_dIW_P = output_dict["gas"]["phase"]["log10dIW_P"]

# Interpolate to find X_O where fO2_IW = 0
X_O_dIW_1_bar = np.interp(0.0, fO2_dIW_1_bar.ravel(), X_O.ravel())
X_O_dIW_P = np.interp(0.0, fO2_dIW_P.ravel(), X_O.ravel())

logger.info("X_O at IW buffer (1 bar): %0.6e", X_O_dIW_1_bar)
logger.info("X_O at IW buffer: %0.6e", X_O_dIW_P)

Plot Figure 2.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

filename = Path("ZD09_fig2a_data.csv")
data = importlib.resources.as_file(
    DATA_DIRECTORY.joinpath(str(ZHANG_DUAN_DIRECTORY.joinpath(filename)))
)
with data as datapath:
    df: pd.DataFrame = pd.read_csv(datapath)

ax[0].plot(df["X_O"], df["X_H2O"], "s", label="Data from ZD09 Fig. 2a")
ax[0].plot(X_O, X_H2O, linestyle="-", color="black", label="Model")

ax[0].set_xlabel("X_O")
ax[0].set_ylabel("X_H2O")
ax[0].set_title("X_H2O vs X_O")
ax[0].set_xlim(0.15, 0.4)
ax[0].set_ylim(0.5, 1.0)
ax[0].legend()

filename: Path = Path("ZD09_fig2b_data.csv")
data = importlib.resources.as_file(
    DATA_DIRECTORY.joinpath(str(ZHANG_DUAN_DIRECTORY.joinpath(filename)))
)
with data as datapath:
    df: pd.DataFrame = pd.read_csv(datapath)

ax[1].plot(df["X_O"], df["X_CH4"], "s", label="Data from ZD09 Fig. 2b")
ax[1].plot(X_O, X_CH4, linestyle="-", color="black", label="Model")
ax[1].set_xlabel("X_O")
ax[1].set_ylabel("X_CH4")
ax[1].set_title("X_CH4 vs X_O")
ax[1].set_xlim(0, 0.15)
ax[1].set_ylim(0.5, 1.0)
ax[1].legend()

fig.suptitle("Comparison of Zhang and Duan (2009) model to data from their Fig. 2")

Plot Figure 3.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 5))

ax[0].plot(X_O, np.log10(fO2), linestyle="-", color="black", label="log(fO2)")
ax[0].axvline(X_O_dIW_P, linestyle=":", color="black", label="IW buffer at P")
ax[0].axvline(X_O_dIW_1_bar, linestyle="--", color="black", label="IW buffer at 1 bar")
ax[0].set_xlim(0, 1)
ax[0].set_ylim(-18, -9)
ax[0].set_ylabel("log(fO2)")

ax[1].plot(X_O, X_H2O, linestyle="-", color="blue", label="H2O")
ax[1].plot(X_O, X_CH4, linestyle="-", color="purple", label="CH4")
ax[1].plot(X_O, X_H2, linestyle="-", color="green", label="H2")
ax[1].plot(X_O, X_CO2, linestyle="-", color="orange", label="CO2")
ax[1].axvline(X_O_dIW_P, linestyle=":", color="black", label="IW buffer at P")
ax[1].axvline(X_O_dIW_1_bar, linestyle="--", color="black", label="IW buffer at 1 bar")
ax[1].set_xlim(0, 1)
ax[1].set_ylim(0, 1)
ax[1].set_xlabel("X_O")
ax[1].set_ylabel("Mole fraction")

ax[1].legend()

fig.suptitle("Predicted composition of the carbon-saturated C-O-H fluid system")
